# 05 — Step 4: Spatial Attention Pattern Analysis

**Goal**: Visualize what candidate reflection heads actually attend to.

For query tokens in the reflection region, show which key tokens they attend to.

In [ ]:
import sys
sys.path.insert(0, '/content/mi-mirror')

import pickle
import matplotlib.pyplot as plt
from PIL import Image

from scripts.config import (
    EXPERIMENTS_DIR, MIRROR_PROMPTS, SEEDS,
    NUM_INFERENCE_STEPS,
)
from scripts.model_loader import load_pipeline
from scripts.roi import get_default_rois
from scripts.generate import generate_with_full_attention
from scripts.attention_extraction import AttentionData
from scripts.visualization import plot_spatial_attention

STEP1_DIR = EXPERIMENTS_DIR / "step1_selectivity"
STEP3_DIR = EXPERIMENTS_DIR / "step3_temporal"
STEP4_DIR = EXPERIMENTS_DIR / "step4_spatial"
STEP4_DIR.mkdir(parents=True, exist_ok=True)

with open(STEP1_DIR / "step1_results.pkl", "rb") as f:
    step1 = pickle.load(f)
candidates = step1["candidates"]
block_infos = step1["block_infos"]
info_map = {b.linear_idx: b for b in block_infos}

with open(STEP3_DIR / "step3_results.pkl", "rb") as f:
    step3 = pickle.load(f)
peak_timestep = step3["peak_timestep"]

candidate_linear_indices = list(set(b for b, h, _ in candidates[:5]))
obj_roi, ref_roi = get_default_rois()

print(f"Top-5 candidates: {[(b, h) for b, h, _ in candidates[:5]]}")
print(f"Peak timestep: {peak_timestep}")

In [ ]:
pipe = load_pipeline()
print("Model loaded.")

In [ ]:
# Generate with full attention extraction on first mirror prompt
prompt = MIRROR_PROMPTS[0]
seed = SEEDS[0]
print(f"Prompt: {prompt}")
print(f"Seed: {seed}")

attn_data = AttentionData()
image, attn_data, _ = generate_with_full_attention(
    pipe, prompt, seed, obj_roi, ref_roi,
    candidate_linear_indices=candidate_linear_indices, attention_data=attn_data,
)

print(f"\nStored {len(attn_data.full_maps)} full attention maps")
print(f"Map keys: {list(attn_data.full_maps.keys())}")

In [ ]:
# Visualize spatial attention for top candidates
for b, h, s in candidates[:5]:
    # Find a stored attention map for this block
    key = (b, peak_timestep)
    if key not in attn_data.full_maps:
        for t in range(NUM_INFERENCE_STEPS):
            if (b, t) in attn_data.full_maps:
                key = (b, t)
                break
        else:
            print(f"No attention map for block {b}, skipping")
            continue

    attn_map = attn_data.full_maps[key]
    info = info_map[b]
    grid_size = info.resolution
    ref_indices = ref_roi.token_indices(grid_size)

    fig = plot_spatial_attention(
        attn_map, head_idx=h,
        query_indices=ref_indices,
        grid_size=grid_size,
        image=image,
        title=f"{info.position} {info.block_idx}.{info.layer_idx}, Head {h} (S={s:.4f}) — Reflection queries",
    )
    fig.savefig(STEP4_DIR / f"spatial_B{b}_H{h}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Reverse direction: object queries attending to reflection region
for b, h, s in candidates[:3]:
    key = (b, peak_timestep)
    if key not in attn_data.full_maps:
        for t in range(NUM_INFERENCE_STEPS):
            if (b, t) in attn_data.full_maps:
                key = (b, t)
                break
        else:
            continue

    attn_map = attn_data.full_maps[key]
    info = info_map[b]
    grid_size = info.resolution
    obj_indices = obj_roi.token_indices(grid_size)

    fig = plot_spatial_attention(
        attn_map, head_idx=h,
        query_indices=obj_indices,
        grid_size=grid_size,
        image=image,
        title=f"{info.position} {info.block_idx}.{info.layer_idx}, Head {h} — Object queries (reverse)",
    )
    fig.savefig(STEP4_DIR / f"spatial_reverse_B{b}_H{h}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
results = {
    "prompt": prompt,
    "seed": seed,
    "candidate_linear_indices": candidate_linear_indices,
    "peak_timestep": peak_timestep,
}
with open(STEP4_DIR / "step4_results.pkl", "wb") as f:
    pickle.dump(results, f)

print(f"Results saved to {STEP4_DIR}")
print("\n=== Step 4 Complete ===")
print("Next: Generate HTML report (report/index.html)")